In [1]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_data_path = project_root / "data" / "processed"
database_path = processed_data_path / "ravenstack.db"

In [3]:
engine = create_engine(f"sqlite:///{database_path}")

print("Database location:")
print(database_path)

Database location:
c:\Users\tOBESky\Documents\ML_BigDATA\saas-customer-churn-prediction\data\processed\ravenstack.db


In [4]:
accounts_sql = pd.read_csv(
    processed_data_path / "ravenstack_accounts_clean.csv",
    parse_dates=["signup_date"]
)

In [5]:
accounts_sql.to_sql(
    name="accounts",
    con=engine,
    if_exists="replace",
    index=False
)

500

In [6]:
accounts_check = pd.read_sql_query(
    """
    SELECT *
    FROM accounts
    LIMIT 5;
    """,
    engine
)

accounts_check

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16 00:00:00.000000,partner,Basic,9,0,0
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17 00:00:00.000000,other,Basic,18,0,1
2,A-0a282f,Company_2,DevTools,US,2024-08-27 00:00:00.000000,organic,Basic,1,0,0
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27 00:00:00.000000,other,Basic,24,1,0
4,A-ce550d,Company_4,HealthTech,US,2024-10-27 00:00:00.000000,event,Enterprise,35,0,1


In [7]:
account_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS total_accounts
    FROM accounts;
    """,
    engine
)

account_count

,total_accounts
0,500


In [8]:
subscriptions_sql = pd.read_csv(
    processed_data_path / "ravenstack_subscriptions_clean.csv",
    parse_dates=["start_date", "end_date"]
)

feature_usage_sql = pd.read_csv(
    processed_data_path / "ravenstack_feature_usage_clean.csv",
    parse_dates=["usage_date"]
)

support_tickets_sql = pd.read_csv(
    processed_data_path / "ravenstack_support_tickets_clean.csv",
    parse_dates=["submitted_at", "closed_at"]
)

churn_events_sql = pd.read_csv(
    processed_data_path / "ravenstack_churn_events_clean.csv",
    parse_dates=["churn_date"]
)

In [9]:
subscriptions_sql.to_sql(
    name="subscriptions",
    con=engine,
    if_exists="replace",
    index=False
)

feature_usage_sql.to_sql(
    name="feature_usage",
    con=engine,
    if_exists="replace",
    index=False
)

support_tickets_sql.to_sql(
    name="support_tickets",
    con=engine,
    if_exists="replace",
    index=False
)

churn_events_sql.to_sql(
    name="churn_events",
    con=engine,
    if_exists="replace",
    index=False
)

600

In [10]:
tables_check = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    engine
)

tables_check

,name
0,accounts
1,churn_events
2,feature_usage
3,subscriptions
4,support_tickets


In [12]:
table_counts = pd.read_sql_query(
    """
    SELECT 'accounts' AS table_name, COUNT(*) AS row_count
    FROM accounts

    UNION ALL

    SELECT 'subscriptions', COUNT(*)
    FROM subscriptions

    UNION ALL

    SELECT 'feature_usage', COUNT(*)
    FROM feature_usage

    UNION ALL

    SELECT 'support_tickets', COUNT(*)
    FROM support_tickets

    UNION ALL

    SELECT 'churn_events', COUNT(*)
    FROM churn_events;
    """,
    engine
)

table_counts

,table_name,row_count
0,accounts,500
1,subscriptions,5000
2,feature_usage,25000
3,support_tickets,2000
4,churn_events,600


What percentage of RavenStack customer accounts have churned?

In [13]:
churn_summary = pd.read_sql_query(
    """
    SELECT
        churn_flag,
        COUNT(*) AS account_count
    FROM accounts
    GROUP BY churn_flag;
    """,
    engine
)

churn_summary

,churn_flag,account_count
0,0,390
1,1,110


In [15]:
churn_rate = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_accounts,
        SUM(churn_flag) AS churned_accounts,
        ROUND(
            100.0 * SUM(churn_flag) / COUNT(*),
            2
        ) AS churn_rate_percent
    FROM accounts;
    """,
    engine
)

churn_rate

,total_accounts,churned_accounts,churn_rate_percent
0,500,110,22.0


In [16]:
churn_by_plan = pd.read_sql_query(
    """
    SELECT
        plan_tier,
        COUNT(*) AS total_accounts,
        SUM(churn_flag) AS churned_accounts,
        ROUND(
            100.0 * SUM(churn_flag) / COUNT(*),
            2
        ) AS churn_rate_percent
    FROM accounts
    GROUP BY plan_tier
    ORDER BY churn_rate_percent DESC;
    """,
    engine
)

churn_by_plan

,plan_tier,total_accounts,churned_accounts,churn_rate_percent
0,Enterprise,154,34,22.08
1,Basic,168,37,22.02
2,Pro,178,39,21.91


In [17]:
churn_by_industry = pd.read_sql_query(
    """
    SELECT
        industry,
        COUNT(*) AS total_accounts,
        SUM(churn_flag) AS churned_accounts,
        ROUND(
            100.0 * SUM(churn_flag) / COUNT(*),
            2
        ) AS churn_rate_percent
    FROM accounts
    GROUP BY industry
    ORDER BY churn_rate_percent DESC;
    """,
    engine
)

churn_by_industry

,industry,total_accounts,churned_accounts,churn_rate_percent
0,DevTools,113,35,30.97
1,FinTech,112,25,22.32
2,HealthTech,96,21,21.88
3,EdTech,79,13,16.46
4,Cybersecurity,100,16,16.00


In [18]:
churn_by_referral = pd.read_sql_query(
    """
    SELECT
        referral_source,
        COUNT(*) AS total_accounts,
        SUM(churn_flag) AS churned_accounts,
        ROUND(
            100.0 * SUM(churn_flag) / COUNT(*),
            2
        ) AS churn_rate_percent
    FROM accounts
    GROUP BY referral_source
    ORDER BY churn_rate_percent DESC;
    """,
    engine
)

churn_by_referral

,referral_source,total_accounts,churned_accounts,churn_rate_percent
0,event,96,29,30.21
1,other,103,25,24.27
2,ads,98,23,23.47
3,organic,114,20,17.54
4,partner,89,13,14.61


In [19]:
churn_by_trial = pd.read_sql_query(
    """
    SELECT
        is_trial,
        COUNT(*) AS total_accounts,
        SUM(churn_flag) AS churned_accounts,
        ROUND(
            100.0 * SUM(churn_flag) / COUNT(*),
            2
        ) AS churn_rate_percent
    FROM accounts
    GROUP BY is_trial
    ORDER BY churn_rate_percent DESC;
    """,
    engine
)

churn_by_trial

,is_trial,total_accounts,churned_accounts,churn_rate_percent
0,1,97,25,25.77
1,0,403,85,21.09


In [20]:
churn_by_country = pd.read_sql_query(
    """
    SELECT
        country,
        COUNT(*) AS total_accounts,
        SUM(churn_flag) AS churned_accounts,
        ROUND(
            100.0 * SUM(churn_flag) / COUNT(*),
            2
        ) AS churn_rate_percent
    FROM accounts
    GROUP BY country
    ORDER BY churn_rate_percent DESC;
    """,
    engine
)

churn_by_country

,country,total_accounts,churned_accounts,churn_rate_percent
0,DE,25,8,32.00
1,US,291,68,23.37
2,FR,22,5,22.73
3,IN,49,10,20.41
4,UK,58,11,18.97
5,CA,23,4,17.39
6,AU,32,4,12.50


In [21]:
seats_by_churn = pd.read_sql_query(
    """
    SELECT
        churn_flag,
        COUNT(*) AS account_count,
        ROUND(AVG(seats), 2) AS average_seats,
        MIN(seats) AS minimum_seats,
        MAX(seats) AS maximum_seats
    FROM accounts
    GROUP BY churn_flag;
    """,
    engine
)

seats_by_churn

,churn_flag,account_count,average_seats,minimum_seats,maximum_seats
0,0,390,20.93,1,163
1,1,110,19.24,1,87


In [22]:
churn_by_signup_year = pd.read_sql_query(
    """
    SELECT
        strftime('%Y', signup_date) AS signup_year,
        COUNT(*) AS total_accounts,
        SUM(churn_flag) AS churned_accounts,
        ROUND(
            100.0 * SUM(churn_flag) / COUNT(*),
            2
        ) AS churn_rate_percent
    FROM accounts
    GROUP BY signup_year
    ORDER BY signup_year;
    """,
    engine
)

churn_by_signup_year

,signup_year,total_accounts,churned_accounts,churn_rate_percent
0,2023,227,58,25.55
1,2024,273,52,19.05


In [23]:
churn_by_account_size = pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN seats <= 10 THEN 'Small (1-10)'
            WHEN seats <= 25 THEN 'Medium (11-25)'
            WHEN seats <= 50 THEN 'Large (26-50)'
            ELSE 'Very Large (51+)'
        END AS account_size,

        COUNT(*) AS total_accounts,
        SUM(churn_flag) AS churned_accounts,

        ROUND(
            100.0 * SUM(churn_flag) / COUNT(*),
            2
        ) AS churn_rate_percent

    FROM accounts

    GROUP BY account_size
    ORDER BY churn_rate_percent DESC;
    """,
    engine
)

churn_by_account_size

,account_size,total_accounts,churned_accounts,churn_rate_percent
0,Small (1-10),198,47,23.74
1,Large (26-50),106,23,21.70
2,Very Large (51+),37,8,21.62
3,Medium (11-25),159,32,20.13


In [24]:
account_subscription_summary = pd.read_sql_query(
    """
    SELECT
        account_id,
        COUNT(*) AS subscription_count,
        MAX(upgrade_flag) AS ever_upgraded,
        MAX(downgrade_flag) AS ever_downgraded,
        MAX(auto_renew_flag) AS has_auto_renew_record
    FROM subscriptions
    GROUP BY account_id;
    """,
    engine
)

account_subscription_summary.head()

,account_id,subscription_count,ever_upgraded,ever_downgraded,has_auto_renew_record
0,A-00bed1,10,1,0,1
1,A-00cac8,9,0,1,1
2,A-0158bb,6,1,0,1
3,A-016043,11,1,0,1
4,A-019782,9,1,0,1


In [25]:
print("Rows in account subscription summary:", len(account_subscription_summary))

Rows in account subscription summary: 500


In [26]:
churn_by_downgrade = pd.read_sql_query(
    """
    WITH subscription_summary AS (
        SELECT
            account_id,
            MAX(downgrade_flag) AS ever_downgraded
        FROM subscriptions
        GROUP BY account_id
    )

    SELECT
        s.ever_downgraded,
        COUNT(*) AS total_accounts,
        SUM(a.churn_flag) AS churned_accounts,
        ROUND(
            100.0 * SUM(a.churn_flag) / COUNT(*),
            2
        ) AS churn_rate_percent
    FROM accounts AS a
    JOIN subscription_summary AS s
        ON a.account_id = s.account_id
    GROUP BY s.ever_downgraded
    ORDER BY churn_rate_percent DESC;
    """,
    engine
)

churn_by_downgrade

,ever_downgraded,total_accounts,churned_accounts,churn_rate_percent
0,0,323,73,22.6
1,1,177,37,20.9


In [27]:
churn_by_upgrade = pd.read_sql_query(
    """
    WITH subscription_summary AS (
        SELECT
            account_id,
            MAX(upgrade_flag) AS ever_upgraded
        FROM subscriptions
        GROUP BY account_id
    )

    SELECT
        s.ever_upgraded,
        COUNT(*) AS total_accounts,
        SUM(a.churn_flag) AS churned_accounts,
        ROUND(
            100.0 * SUM(a.churn_flag) / COUNT(*),
            2
        ) AS churn_rate_percent

    FROM accounts AS a

    JOIN subscription_summary AS s
        ON a.account_id = s.account_id

    GROUP BY s.ever_upgraded

    ORDER BY churn_rate_percent DESC;
    """,
    engine
)

churn_by_upgrade

,ever_upgraded,total_accounts,churned_accounts,churn_rate_percent
0,0,193,46,23.83
1,1,307,64,20.85


In [28]:
subscription_count_by_churn = pd.read_sql_query(
    """
    WITH subscription_summary AS (
        SELECT
            account_id,
            COUNT(*) AS subscription_count
        FROM subscriptions
        GROUP BY account_id
    )

    SELECT
        a.churn_flag,
        COUNT(*) AS total_accounts,
        ROUND(AVG(s.subscription_count), 2) AS average_subscription_count,
        MIN(s.subscription_count) AS minimum_subscription_count,
        MAX(s.subscription_count) AS maximum_subscription_count

    FROM accounts AS a

    JOIN subscription_summary AS s
        ON a.account_id = s.account_id

    GROUP BY a.churn_flag;
    """,
    engine
)

subscription_count_by_churn

,churn_flag,total_accounts,average_subscription_count,minimum_subscription_count,maximum_subscription_count
0,0,390,9.92,2,19
1,1,110,10.29,3,19


In [29]:
subscription_date_range = pd.read_sql_query(
    """
    SELECT
        MIN(start_date) AS earliest_subscription,
        MAX(start_date) AS latest_subscription,
        MIN(end_date) AS earliest_end_date,
        MAX(end_date) AS latest_end_date
    FROM subscriptions;
    """,
    engine
)

subscription_date_range

,earliest_subscription,latest_subscription,earliest_end_date,latest_end_date
0,2023-01-09 00:00:00.000000,2024-12-31 00:00:00.000000,2023-04-05 00:00:00.000000,2024-12-31 00:00:00.000000


In [31]:
latest_subscriptions = pd.read_sql_query(
    """
    WITH ranked_subscriptions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY account_id
                ORDER BY start_date DESC
            ) AS row_number
        FROM subscriptions
    )

    SELECT *
    FROM ranked_subscriptions
    WHERE row_number = 1;
    """,
    engine
)

latest_subscriptions.head()

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag,row_number
0,S-289702,A-00bed1,2024-09-01 00:00:00.000000,NaN,Enterprise,28,5572,66864,0,0,0,0,annual,0,1
1,S-c0ca82,A-00cac8,2024-12-02 00:00:00.000000,NaN,Pro,19,931,11172,0,0,1,0,annual,1,1
2,S-fb9972,A-0158bb,2024-12-31 00:00:00.000000,NaN,Pro,45,2205,26460,0,1,0,0,annual,1,1
3,S-463525,A-016043,2024-12-02 00:00:00.000000,NaN,Enterprise,13,2587,31044,0,0,0,0,annual,1,1
4,S-2fd650,A-019782,2024-12-04 00:00:00.000000,NaN,Enterprise,15,0,0,1,0,0,0,annual,0,1


In [32]:
print("Latest subscription rows:", len(latest_subscriptions))
print("Unique accounts:", latest_subscriptions["account_id"].nunique())

Latest subscription rows: 500
Unique accounts: 500


In [33]:
churn_by_auto_renew = pd.read_sql_query(
    """
    WITH ranked_subscriptions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY account_id
                ORDER BY start_date DESC
            ) AS row_number
        FROM subscriptions
    ),

    latest_subscription AS (
        SELECT
            account_id,
            auto_renew_flag
        FROM ranked_subscriptions
        WHERE row_number = 1
    )

    SELECT
        l.auto_renew_flag,
        COUNT(*) AS total_accounts,
        SUM(a.churn_flag) AS churned_accounts,
        ROUND(
            100.0 * SUM(a.churn_flag) / COUNT(*),
            2
        ) AS churn_rate_percent

    FROM accounts AS a

    JOIN latest_subscription AS l
        ON a.account_id = l.account_id

    GROUP BY l.auto_renew_flag

    ORDER BY churn_rate_percent DESC;
    """,
    engine
)

churn_by_auto_renew

,auto_renew_flag,total_accounts,churned_accounts,churn_rate_percent
0,0,86,19,22.09
1,1,414,91,21.98


In [34]:
churn_by_billing_frequency = pd.read_sql_query(
    """
    WITH ranked_subscriptions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY account_id
                ORDER BY start_date DESC
            ) AS row_number
        FROM subscriptions
    ),

    latest_subscription AS (
        SELECT
            account_id,
            billing_frequency
        FROM ranked_subscriptions
        WHERE row_number = 1
    )

    SELECT
        l.billing_frequency,
        COUNT(*) AS total_accounts,
        SUM(a.churn_flag) AS churned_accounts,
        ROUND(
            100.0 * SUM(a.churn_flag) / COUNT(*),
            2
        ) AS churn_rate_percent

    FROM accounts AS a

    JOIN latest_subscription AS l
        ON a.account_id = l.account_id

    GROUP BY l.billing_frequency
    ORDER BY churn_rate_percent DESC;
    """,
    engine
)

churn_by_billing_frequency

,billing_frequency,total_accounts,churned_accounts,churn_rate_percent
0,annual,281,68,24.20
1,monthly,219,42,19.18


In [35]:
mrr_by_churn = pd.read_sql_query(
    """
    WITH ranked_subscriptions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY account_id
                ORDER BY start_date DESC
            ) AS row_number
        FROM subscriptions
    ),

    latest_subscription AS (
        SELECT
            account_id,
            mrr_amount
        FROM ranked_subscriptions
        WHERE row_number = 1
    )

    SELECT
        a.churn_flag,
        COUNT(*) AS total_accounts,
        ROUND(AVG(l.mrr_amount), 2) AS average_mrr,
        MIN(l.mrr_amount) AS minimum_mrr,
        MAX(l.mrr_amount) AS maximum_mrr

    FROM accounts AS a

    JOIN latest_subscription AS l
        ON a.account_id = l.account_id

    GROUP BY a.churn_flag;
    """,
    engine
)

mrr_by_churn

,churn_flag,total_accounts,average_mrr,minimum_mrr,maximum_mrr
0,0,390,2520.38,0,33830
1,1,110,2322.20,0,21691


In [36]:
churn_by_mrr_band = pd.read_sql_query(
    """
    WITH ranked_subscriptions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY account_id
                ORDER BY start_date DESC
            ) AS row_number
        FROM subscriptions
    ),

    latest_subscription AS (
        SELECT
            account_id,
            mrr_amount
        FROM ranked_subscriptions
        WHERE row_number = 1
    ),

    mrr_groups AS (
        SELECT
            account_id,
            CASE
                WHEN mrr_amount = 0 THEN 'No MRR'
                WHEN mrr_amount <= 1000 THEN 'Low (1-1000)'
                WHEN mrr_amount <= 3000 THEN 'Medium (1001-3000)'
                WHEN mrr_amount <= 5000 THEN 'High (3001-5000)'
                ELSE 'Very High (5001+)'
            END AS mrr_band
        FROM latest_subscription
    )

    SELECT
        m.mrr_band,
        COUNT(*) AS total_accounts,
        SUM(a.churn_flag) AS churned_accounts,
        ROUND(
            100.0 * SUM(a.churn_flag) / COUNT(*),
            2
        ) AS churn_rate_percent

    FROM accounts AS a

    JOIN mrr_groups AS m
        ON a.account_id = m.account_id

    GROUP BY m.mrr_band
    ORDER BY churn_rate_percent DESC;
    """,
    engine
)

churn_by_mrr_band

,mrr_band,total_accounts,churned_accounts,churn_rate_percent
0,High (3001-5000),53,14,26.42
1,Medium (1001-3000),120,28,23.33
2,Low (1-1000),176,38,21.59
3,Very High (5001+),75,16,21.33
4,No MRR,76,14,18.42


In [37]:
zero_mrr_trial_check = pd.read_sql_query(
    """
    WITH ranked_subscriptions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY account_id
                ORDER BY start_date DESC
            ) AS row_number
        FROM subscriptions
    ),

    latest_subscription AS (
        SELECT
            account_id,
            mrr_amount,
            is_trial
        FROM ranked_subscriptions
        WHERE row_number = 1
    )

    SELECT
        is_trial,
        COUNT(*) AS total_accounts,
        SUM(
            CASE
                WHEN mrr_amount = 0 THEN 1
                ELSE 0
            END
        ) AS zero_mrr_accounts

    FROM latest_subscription

    GROUP BY is_trial;
    """,
    engine
)

zero_mrr_trial_check

,is_trial,total_accounts,zero_mrr_accounts
0,0,424,0
1,1,76,76


In [38]:
account_vs_latest_trial = pd.read_sql_query(
    """
    WITH ranked_subscriptions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY account_id
                ORDER BY start_date DESC
            ) AS row_number
        FROM subscriptions
    ),

    latest_subscription AS (
        SELECT
            account_id,
            is_trial AS latest_subscription_trial
        FROM ranked_subscriptions
        WHERE row_number = 1
    )

    SELECT
        a.is_trial AS account_trial,
        l.latest_subscription_trial,
        COUNT(*) AS total_accounts

    FROM accounts AS a

    JOIN latest_subscription AS l
        ON a.account_id = l.account_id

    GROUP BY
        a.is_trial,
        l.latest_subscription_trial

    ORDER BY
        a.is_trial,
        l.latest_subscription_trial;
    """,
    engine
)

account_vs_latest_trial

,account_trial,latest_subscription_trial,total_accounts
0,0,0,342
1,0,1,61
2,1,0,82
3,1,1,15


In [39]:
trial_status_mismatches = pd.read_sql_query(
    """
    WITH ranked_subscriptions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY account_id
                ORDER BY start_date DESC
            ) AS row_number
        FROM subscriptions
    ),

    latest_subscription AS (
        SELECT
            account_id,
            start_date,
            end_date,
            plan_tier,
            mrr_amount,
            is_trial AS latest_subscription_trial
        FROM ranked_subscriptions
        WHERE row_number = 1
    )

    SELECT
        a.account_id,
        a.is_trial AS account_trial,
        l.latest_subscription_trial,
        l.start_date,
        l.end_date,
        l.plan_tier,
        l.mrr_amount

    FROM accounts AS a

    JOIN latest_subscription AS l
        ON a.account_id = l.account_id

    WHERE a.is_trial != l.latest_subscription_trial

    LIMIT 15;
    """,
    engine
)

trial_status_mismatches

,account_id,account_trial,latest_subscription_trial,start_date,end_date,plan_tier,mrr_amount
0,A-1f0ac7,1,0,2024-12-20 00:00:00.000000,NaN,Pro,1176
1,A-1b9609,0,1,2024-10-03 00:00:00.000000,NaN,Enterprise,0
2,A-7dacce,0,1,2024-12-04 00:00:00.000000,NaN,Pro,0
3,A-462d45,1,0,2024-12-10 00:00:00.000000,NaN,Pro,1470
4,A-832ec2,1,0,2024-11-27 00:00:00.000000,NaN,Enterprise,4378
5,A-684255,0,1,2024-12-15 00:00:00.000000,2024-12-26 00:00:00.000000,Basic,0
6,A-9f2731,0,1,2024-11-22 00:00:00.000000,NaN,Pro,0
7,A-00cac8,1,0,2024-12-02 00:00:00.000000,NaN,Pro,931
8,A-8145a0,1,0,2024-12-17 00:00:00.000000,NaN,Basic,285
9,A-44dc83,0,1,2024-11-26 00:00:00.000000,NaN,Enterprise,0


In [40]:
plan_status_check = pd.read_sql_query(
    """
    WITH ranked_subscriptions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY account_id
                ORDER BY start_date DESC
            ) AS row_number
        FROM subscriptions
    ),

    latest_subscription AS (
        SELECT
            account_id,
            plan_tier AS latest_plan_tier
        FROM ranked_subscriptions
        WHERE row_number = 1
    )

    SELECT
        a.plan_tier AS account_plan_tier,
        l.latest_plan_tier,
        COUNT(*) AS total_accounts

    FROM accounts AS a

    JOIN latest_subscription AS l
        ON a.account_id = l.account_id

    GROUP BY
        a.plan_tier,
        l.latest_plan_tier

    ORDER BY
        a.plan_tier,
        l.latest_plan_tier;
    """,
    engine
)

plan_status_check

,account_plan_tier,latest_plan_tier,total_accounts
0,Basic,Basic,54
1,Basic,Enterprise,57
2,Basic,Pro,57
3,Enterprise,Basic,47
4,Enterprise,Enterprise,59
5,Enterprise,Pro,48
6,Pro,Basic,59
7,Pro,Enterprise,60
8,Pro,Pro,59


In [41]:
churn_by_latest_plan = pd.read_sql_query(
    """
    WITH ranked_subscriptions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY account_id
                ORDER BY start_date DESC
            ) AS row_number
        FROM subscriptions
    ),

    latest_subscription AS (
        SELECT
            account_id,
            plan_tier AS latest_plan_tier
        FROM ranked_subscriptions
        WHERE row_number = 1
    )

    SELECT
        l.latest_plan_tier,
        COUNT(*) AS total_accounts,
        SUM(a.churn_flag) AS churned_accounts,
        ROUND(
            100.0 * SUM(a.churn_flag) / COUNT(*),
            2
        ) AS churn_rate_percent

    FROM accounts AS a

    JOIN latest_subscription AS l
        ON a.account_id = l.account_id

    GROUP BY l.latest_plan_tier

    ORDER BY churn_rate_percent DESC;
    """,
    engine
)

churn_by_latest_plan

,latest_plan_tier,total_accounts,churned_accounts,churn_rate_percent
0,Enterprise,176,41,23.30
1,Basic,160,35,21.88
2,Pro,164,34,20.73


In [42]:
latest_seats_by_churn = pd.read_sql_query(
    """
    WITH ranked_subscriptions AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY account_id
                ORDER BY start_date DESC
            ) AS row_number
        FROM subscriptions
    ),

    latest_subscription AS (
        SELECT
            account_id,
            seats
        FROM ranked_subscriptions
        WHERE row_number = 1
    )

    SELECT
        a.churn_flag,
        COUNT(*) AS total_accounts,
        ROUND(AVG(l.seats), 2) AS average_latest_seats,
        MIN(l.seats) AS minimum_latest_seats,
        MAX(l.seats) AS maximum_latest_seats

    FROM accounts AS a

    JOIN latest_subscription AS l
        ON a.account_id = l.account_id

    GROUP BY a.churn_flag;
    """,
    engine
)

latest_seats_by_churn

,churn_flag,total_accounts,average_latest_seats,minimum_latest_seats,maximum_latest_seats
0,0,390,30.82,1,170
1,1,110,28.22,3,109


In [43]:
mrr_arr_relationship = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_subscriptions,

        SUM(
            CASE
                WHEN arr_amount = mrr_amount * 12 THEN 1
                ELSE 0
            END
        ) AS matching_records,

        SUM(
            CASE
                WHEN arr_amount != mrr_amount * 12 THEN 1
                ELSE 0
            END
        ) AS non_matching_records

    FROM subscriptions;
    """,
    engine
)

mrr_arr_relationship

,total_subscriptions,matching_records,non_matching_records
0,5000,5000,0


In [44]:
account_usage_summary = pd.read_sql_query(
    """
    SELECT
        s.account_id,

        COUNT(*) AS usage_event_count,

        SUM(f.usage_count) AS total_usage_count,

        ROUND(
            AVG(f.usage_count),
            2
        ) AS average_usage_count,

        SUM(f.usage_duration_secs) AS total_usage_duration_secs,

        SUM(f.error_count) AS total_errors,

        COUNT(DISTINCT f.feature_name) AS unique_features_used,

        COUNT(DISTINCT DATE(f.usage_date)) AS active_usage_days,

        MAX(f.usage_date) AS last_usage_date,

        SUM(
            CASE
                WHEN f.is_beta_feature = 1 THEN 1
                ELSE 0
            END
        ) AS beta_usage_events

    FROM feature_usage AS f

    JOIN subscriptions AS s
        ON f.subscription_id = s.subscription_id

    GROUP BY s.account_id;
    """,
    engine
)

account_usage_summary.head()

,account_id,usage_event_count,total_usage_count,average_usage_count,total_usage_duration_secs,total_errors,unique_features_used,active_usage_days,last_usage_date,beta_usage_events
0,A-00bed1,51,514,10.08,143734,27,32,49,2024-12-14 00:00:00.000000,2
1,A-00cac8,58,602,10.38,171366,31,30,57,2024-12-24 00:00:00.000000,7
2,A-0158bb,36,364,10.11,122051,22,19,35,2024-12-28 00:00:00.000000,3
3,A-016043,47,490,10.43,132075,21,26,46,2024-12-31 00:00:00.000000,3
4,A-019782,55,562,10.22,160848,30,28,55,2024-12-20 00:00:00.000000,3


In [45]:
usage_by_churn = pd.read_sql_query(
    """
    WITH usage_summary AS (
        SELECT
            s.account_id,

            COUNT(*) AS usage_event_count,

            SUM(f.usage_count) AS total_usage_count,

            SUM(f.usage_duration_secs) AS total_usage_duration_secs,

            SUM(f.error_count) AS total_errors,

            COUNT(DISTINCT f.feature_name) AS unique_features_used,

            COUNT(DISTINCT DATE(f.usage_date)) AS active_usage_days

        FROM feature_usage AS f

        JOIN subscriptions AS s
            ON f.subscription_id = s.subscription_id

        GROUP BY s.account_id
    )

    SELECT
        a.churn_flag,
        COUNT(*) AS total_accounts,

        ROUND(AVG(u.usage_event_count), 2)
            AS average_usage_events,

        ROUND(AVG(u.total_usage_count), 2)
            AS average_total_usage,

        ROUND(AVG(u.total_usage_duration_secs), 2)
            AS average_usage_duration,

        ROUND(AVG(u.total_errors), 2)
            AS average_errors,

        ROUND(AVG(u.unique_features_used), 2)
            AS average_unique_features,

        ROUND(AVG(u.active_usage_days), 2)
            AS average_active_days

    FROM accounts AS a

    JOIN usage_summary AS u
        ON a.account_id = u.account_id

    GROUP BY a.churn_flag;
    """,
    engine
)

usage_by_churn

,churn_flag,total_accounts,average_usage_events,average_total_usage,average_usage_duration,average_errors,average_unique_features,average_active_days
0,0,390,49.42,495.13,150350.88,28.23,27.41,47.69
1,1,110,52.05,522.04,158347.54,28.15,28.34,50.11


In [46]:
usage_quality_by_churn = pd.read_sql_query(
    """
    WITH usage_summary AS (
        SELECT
            s.account_id,

            COUNT(*) AS usage_events,

            SUM(f.usage_count) AS total_usage,

            SUM(f.usage_duration_secs) AS total_duration,

            SUM(f.error_count) AS total_errors

        FROM feature_usage AS f

        JOIN subscriptions AS s
            ON f.subscription_id = s.subscription_id

        GROUP BY s.account_id
    )

    SELECT
        a.churn_flag,
        COUNT(*) AS total_accounts,

        ROUND(
            AVG(1.0 * u.total_usage / u.usage_events),
            2
        ) AS average_usage_per_event,

        ROUND(
            AVG(1.0 * u.total_duration / u.usage_events),
            2
        ) AS average_duration_per_event,

        ROUND(
            AVG(100.0 * u.total_errors / NULLIF(u.total_usage, 0)),
            2
        ) AS average_errors_per_100_usage

    FROM accounts AS a

    JOIN usage_summary AS u
        ON a.account_id = u.account_id

    GROUP BY a.churn_flag;
    """,
    engine
)

usage_quality_by_churn

,churn_flag,total_accounts,average_usage_per_event,average_duration_per_event,average_errors_per_100_usage
0,0,390,10.02,3056.90,5.72
1,1,110,10.02,3054.28,5.39


In [47]:
usage_recency_by_churn = pd.read_sql_query(
    """
    WITH dataset_end AS (
        SELECT
            MAX(usage_date) AS snapshot_date
        FROM feature_usage
    ),

    last_usage AS (
        SELECT
            s.account_id,
            MAX(f.usage_date) AS last_usage_date
        FROM feature_usage AS f

        JOIN subscriptions AS s
            ON f.subscription_id = s.subscription_id

        GROUP BY s.account_id
    )

    SELECT
        a.churn_flag,
        COUNT(*) AS total_accounts,

        ROUND(
            AVG(
                julianday(d.snapshot_date)
                - julianday(l.last_usage_date)
            ),
            2
        ) AS average_days_since_last_usage,

        MIN(
            julianday(d.snapshot_date)
            - julianday(l.last_usage_date)
        ) AS minimum_days_since_last_usage,

        MAX(
            julianday(d.snapshot_date)
            - julianday(l.last_usage_date)
        ) AS maximum_days_since_last_usage

    FROM accounts AS a

    JOIN last_usage AS l
        ON a.account_id = l.account_id

    CROSS JOIN dataset_end AS d

    GROUP BY a.churn_flag;
    """,
    engine
)

usage_recency_by_churn

,churn_flag,total_accounts,average_days_since_last_usage,minimum_days_since_last_usage,maximum_days_since_last_usage
0,0,390,17.45,0.0,136.0
1,1,110,12.27,0.0,71.0


In [48]:
support_by_churn = pd.read_sql_query(
    """
    WITH support_summary AS (
        SELECT
            account_id,

            COUNT(*) AS ticket_count,

            ROUND(
                AVG(first_response_time_minutes),
                2
            ) AS average_first_response_minutes,

            ROUND(
                AVG(resolution_time_hours),
                2
            ) AS average_resolution_hours,

            ROUND(
                AVG(satisfaction_score),
                2
            ) AS average_satisfaction_score,

            SUM(escalation_flag) AS escalation_count

        FROM support_tickets

        GROUP BY account_id
    )

    SELECT
        a.churn_flag,
        COUNT(*) AS total_accounts,

        ROUND(AVG(s.ticket_count), 2)
            AS average_ticket_count,

        ROUND(AVG(s.average_first_response_minutes), 2)
            AS average_first_response_minutes,

        ROUND(AVG(s.average_resolution_hours), 2)
            AS average_resolution_hours,

        ROUND(AVG(s.average_satisfaction_score), 2)
            AS average_satisfaction_score,

        ROUND(AVG(s.escalation_count), 2)
            AS average_escalations

    FROM accounts AS a

    JOIN support_summary AS s
        ON a.account_id = s.account_id

    GROUP BY a.churn_flag;
    """,
    engine
)

support_by_churn

,churn_flag,total_accounts,average_ticket_count,average_first_response_minutes,average_resolution_hours,average_satisfaction_score,average_escalations
0,0,384,4.08,89.58,36.45,3.95,0.18
1,1,108,4.00,84.93,35.49,4.00,0.22


In [49]:
support_by_churn_complete = pd.read_sql_query(
    """
    WITH support_summary AS (
        SELECT
            account_id,

            COUNT(*) AS ticket_count,

            AVG(first_response_time_minutes)
                AS average_first_response_minutes,

            AVG(resolution_time_hours)
                AS average_resolution_hours,

            AVG(satisfaction_score)
                AS average_satisfaction_score,

            SUM(escalation_flag)
                AS escalation_count

        FROM support_tickets
        GROUP BY account_id
    )

    SELECT
        a.churn_flag,

        COUNT(*) AS total_accounts,

        ROUND(
            AVG(COALESCE(s.ticket_count, 0)),
            2
        ) AS average_ticket_count,

        SUM(
            CASE
                WHEN s.account_id IS NULL THEN 1
                ELSE 0
            END
        ) AS accounts_without_tickets,

        ROUND(
            AVG(s.average_first_response_minutes),
            2
        ) AS average_first_response_minutes,

        ROUND(
            AVG(s.average_resolution_hours),
            2
        ) AS average_resolution_hours,

        ROUND(
            AVG(s.average_satisfaction_score),
            2
        ) AS average_satisfaction_score,

        ROUND(
            AVG(COALESCE(s.escalation_count, 0)),
            2
        ) AS average_escalations

    FROM accounts AS a

    LEFT JOIN support_summary AS s
        ON a.account_id = s.account_id

    GROUP BY a.churn_flag;
    """,
    engine
)

support_by_churn_complete

,churn_flag,total_accounts,average_ticket_count,accounts_without_tickets,average_first_response_minutes,average_resolution_hours,average_satisfaction_score,average_escalations
0,0,390,4.02,6,89.58,36.45,3.95,0.18
1,1,110,3.93,2,84.93,35.49,4.00,0.22


In [50]:
churn_event_summary = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_churn_events,
        COUNT(DISTINCT account_id) AS unique_accounts,
        ROUND(AVG(refund_amount_usd), 2) AS average_refund_usd,
        ROUND(SUM(refund_amount_usd), 2) AS total_refund_usd,
        SUM(is_reactivation) AS reactivation_events,
        ROUND(
            100.0 * SUM(is_reactivation) / COUNT(*),
            2
        ) AS reactivation_rate_percent
    FROM churn_events;
    """,
    engine
)

churn_event_summary

,total_churn_events,unique_accounts,average_refund_usd,total_refund_usd,reactivation_events,reactivation_rate_percent
0,600,352,14.42,8652.25,61,10.17


In [51]:
churn_reasons = pd.read_sql_query(
    """
    SELECT
        reason_code,
        COUNT(*) AS churn_events,
        COUNT(DISTINCT account_id) AS unique_accounts,
        ROUND(AVG(refund_amount_usd), 2) AS average_refund_usd,
        ROUND(SUM(refund_amount_usd), 2) AS total_refund_usd,
        SUM(is_reactivation) AS reactivation_events

    FROM churn_events

    GROUP BY reason_code

    ORDER BY churn_events DESC;
    """,
    engine
)

churn_reasons

,reason_code,churn_events,unique_accounts,average_refund_usd,total_refund_usd,reactivation_events
0,features,114,104,16.72,1905.90,13
1,support,104,95,11.73,1219.64,6
2,budget,104,94,12.00,1247.54,12
3,unknown,95,86,18.34,1742.39,9
4,competitor,92,79,13.08,1203.53,12
5,pricing,91,86,14.65,1333.25,9


In [52]:
churn_event_target_check = pd.read_sql_query(
    """
    SELECT
        a.churn_flag,
        COUNT(DISTINCT c.account_id) AS accounts_with_churn_events

    FROM churn_events AS c

    JOIN accounts AS a
        ON c.account_id = a.account_id

    GROUP BY a.churn_flag;
    """,
    engine
)

churn_event_target_check


,churn_flag,accounts_with_churn_events
0,0,277
1,1,75
